In [1]:
5

5

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Integrated\AppData\Local\Temp\ipykernel_12344\4211815356.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from langchain_community.vectorstores import Chroma

persist_directory = "persist_directory3"

vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

print("Vectorstore loaded successfully.")

C:\Users\Integrated\AppData\Local\Temp\ipykernel_12344\896937473.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Vectorstore loaded successfully.


In [4]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Get full docs once
store_data = vectorstore.get()
all_texts = store_data["documents"]
all_metadatas = store_data.get("metadatas", [{}] * len(all_texts))

# Build Document objects with metadata
documents = [
    Document(page_content=text, metadata=meta)
    for text, meta in zip(all_texts, all_metadatas)
]

# Dense retriever (larger k for candidate pool)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# BM25 retriever
bm25 = BM25Retriever.from_documents(documents)
bm25.k = 20


def hybrid_retrieve(query, k_dense=20, k_bm25=20, max_chunks=30):
    
    # Update k dynamically if needed
    dense_retriever.search_kwargs["k"] = k_dense
    bm25.k = k_bm25
    
    # Dense retrieval
    dense_docs = dense_retriever.invoke(query)
    
    # Sparse retrieval
    bm25_docs = bm25.invoke(query)
    
    # Combine
    combined = dense_docs + bm25_docs
    
    # Deduplicate by content
    seen = set()
    unique_docs = []
    
    for doc in combined:
        if doc.page_content not in seen:
            unique_docs.append(doc)
            seen.add(doc.page_content)
    
    return unique_docs[:max_chunks]

In [5]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def hybrid_rerank(query, top_k_final=5):
    
    candidate_docs = hybrid_retrieve(query)
    
    pairs = [(query, doc.page_content) for doc in candidate_docs]
    scores = reranker.predict(pairs)
    
    reranked = sorted(
        zip(candidate_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )
    
    return [doc for doc, score in reranked[:top_k_final]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
template = """
            You are a helpful AI research assistant.

            Use ONLY the provided context to answer the question.
            If the answer is not explicitly present in the context, say:
            "I don't know based on the provided context."



            Give the answer in detail, search if necessary.

            Context:
            {context}

            Question:
            {question}

            Answer:
            """

In [8]:
def rag_pipeline(query):
    
    top_docs = hybrid_rerank(query, top_k_final=5)

    contexts = [doc.page_content for doc in top_docs]

    context_text = "\n\n".join(contexts)

    prompt = template.format(context=context_text, question=query)

    answer = client2.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    ).text

    return {
        "answer": answer,
        "contexts": contexts   # 🔥 REQUIRED FOR RAGAS
    }

In [11]:
query = "Who are the authors of the paper?"
result = rag_pipeline(query)
print("Answer:", result["answer"])

Answer: The authors of the paper are Dhruv Patel, Aran Vyas, and Prof. Hirenkumar Mer.


In [21]:
import json
from datasets import Dataset

with open("D:\Adrta\eval.json", "r") as f:
    reference_data = json.load(f)["qa_pairs"]

ragas_data = []

for item in reference_data:
    question = item["question"]
    ground_truth = item["answer"]

    result = rag_pipeline(question)

    ragas_data.append({
        "question": question,
        "answer": result["answer"],
        "contexts": result["contexts"],
        "ground_truth": ground_truth
    })

dataset = Dataset.from_list(ragas_data)

<>:4: SyntaxWarning: invalid escape sequence '\A'
<>:4: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Integrated\AppData\Local\Temp\ipykernel_3556\2916440434.py:4: SyntaxWarning: invalid escape sequence '\A'
  with open("D:\Adrta\eval.json", "r") as f:


In [40]:
print(dataset[0])

{'question': 'What simulator was used to generate the initial synthetic rocket data?', 'answer': 'OpenRocket was used to generate the initial synthetic rocket data.', 'contexts': ['often remain proprietary. To address this limitation, we \ngenerated a novel synthetic dataset, partially constructed \nusing OpenRocket—a well-known open-source rocket flight \nsimulator—and further extended using domain-informed data \naugmentation techniques. This approach ensures that the \ndataset retains aerodynamic and physical realism while \nproviding the scale necessary for training robust machine \nlearning models. \nOur methodology incorporates features derived from both \naerodynamic properties (e.g., center of pressure, center of \ngravity, thrust, burn time) and environmental factors (e.g., \nwind speed, temperature, humidity) to predict whether a \ngiven rocket configuration is likely to be stable or unstable \nunder launch conditions. This predictive framework has \nsignificant implications 

# RAG Evaluation - W/O RAGAS

In [39]:
def exact_match(pred, truth):
    return int(pred.strip().lower() == truth.strip().lower())

em_scores = []

for item in dataset:
    em_scores.append(exact_match(item["answer"], item["ground_truth"]))

print("Exact Match Accuracy:", sum(em_scores)/len(em_scores))

Exact Match Accuracy: 0.5


In [35]:
from collections import Counter

def compute_f1(pred, truth):
    pred_tokens = pred.lower().split()
    truth_tokens = truth.lower().split()

    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)

    return 2 * precision * recall / (precision + recall)

f1_scores = []

for item in dataset:
    f1_scores.append(compute_f1(item["answer"], item["ground_truth"]))

print("Average F1:", sum(f1_scores)/len(f1_scores))

Average F1: 0.6428571428571428


In [43]:
from collections import Counter

def context_precision(contexts, ground_truth):
    context_text = " ".join(contexts).lower().split()
    gt_tokens = ground_truth.lower().split()

    common = Counter(context_text) & Counter(gt_tokens)
    overlap = sum(common.values())

    return overlap / len(gt_tokens)

cp_scores = []

for item in dataset:
    cp_scores.append(
        context_precision(item["contexts"], item["ground_truth"])
    )

print("Context Precision:", sum(cp_scores)/len(cp_scores))

Context Precision: 0.8083333333333333


In [37]:
def hallucination_check(answer, contexts):
    context_text = " ".join(contexts).lower()
    answer_text = answer.lower()

    unsupported = 0
    total = len(answer_text.split())

    for word in answer_text.split():
        if word not in context_text:
            unsupported += 1

    return unsupported / total

hallucination_rates = []

for item in dataset:
    hallucination_rates.append(
        hallucination_check(item["answer"], item["contexts"])
    )

print("Average Hallucination Rate:", sum(hallucination_rates)/len(hallucination_rates))

Average Hallucination Rate: 0.05


In [38]:
print("Exact Match:", sum(em_scores)/len(em_scores))
print("F1 Score:", sum(f1_scores)/len(f1_scores))
print("Context Precision:", sum(cp_scores)/len(cp_scores))
print("Hallucination Rate:", sum(hallucination_rates)/len(hallucination_rates))

Exact Match: 0.5
F1 Score: 0.6428571428571428
Context Precision: 0.0
Hallucination Rate: 0.05


In [44]:
for item in dataset:
    print("Q:", item["question"])
    print("GT:", item["ground_truth"])
    print("Retrieved snippet sample:", item["contexts"][0][:200])
    print("------")

Q: What simulator was used to generate the initial synthetic rocket data?
GT: OpenRocket was used to generate the initial synthetic rocket data.
Retrieved snippet sample: often remain proprietary. To address this limitation, we 
generated a novel synthetic dataset, partially constructed 
using OpenRocket—a well-known open-source rocket flight 
simulator—and further ext
------
Q: What was the loss function used to train the neural network?
GT: The neural network was trained using Binary Cross-Entropy as the loss function.
Retrieved snippet sample: The model was compiled with: 
 
Loss Function: Binary Cross-Entropy 
 
Optimizer: Adam 
 
Evaluation Metric: Accuracy and F1-Score 
Dropout regularization (0.2 to 0.5) can be applied to reduce 
ove
------


# LLM Evaluation - W/O RAGAS

In [55]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [58]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [59]:
semantic_scores = []

for item in dataset:
    emb_pred = model.encode(item["answer"])
    emb_gt = model.encode(item["ground_truth"])
    
    score = cosine_similarity(emb_pred, emb_gt)
    semantic_scores.append(score)

print("Average Semantic Similarity:", sum(semantic_scores)/len(semantic_scores))

Average Semantic Similarity: 0.79757154


In [60]:
faith_scores = []

for item in dataset:
    context_text = " ".join(item["contexts"])
    
    emb_ans = model.encode(item["answer"])
    emb_ctx = model.encode(context_text)
    
    score = cosine_similarity(emb_ans, emb_ctx)
    faith_scores.append(score)

print("Average Faithfulness:", sum(faith_scores)/len(faith_scores))

Average Faithfulness: 0.30873603


In [61]:
context_recall_scores = []

for item in dataset:
    context_text = " ".join(item["contexts"])
    
    emb_gt = model.encode(item["ground_truth"])
    emb_ctx = model.encode(context_text)
    
    score = cosine_similarity(emb_gt, emb_ctx)
    context_recall_scores.append(score)

print("Average Context Recall:", sum(context_recall_scores)/len(context_recall_scores))

Average Context Recall: 0.4258312


In [ ]:
for i, item in enumerate(dataset):
    print(f"Example {i+1}:")
    print("Question:", item["question"])
    print("Ground Truth:", item["ground_truth"])
    print("Answer:", item["answer"])
    print("Contexts:", item["contexts"])
    print("Semantic Similarity:", semantic_scores[i])
    print("Faithfulness Score:", faith_scores[i])
    print("Context Recall Score:", context_recall_scores[i])
    print("-" * 50)